<a href="https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one client on one report date. I will use March 2026 as my working month.

I will use the warehouse table that contains the monthly content performance data for my lane.

I will use March 2026 as my working month. I chose a middle month instead of the final month so that I am not developing the logic on the sealed outcome month.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

I will use a small number of performance fields that are available before making the prediction.

### Label

My label is whether the content page is declining or not declining.

### Context

The client, content item, and month/date are context fields. They help identify the row and its time period.

### Excluded

I will exclude fields that are used to create the decline label. I will also exclude fields that would only be known after the prediction time because they could cause data leakage.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Warehouse connection ready")

Warehouse connection ready


In [17]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [18]:
sample = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
""").df()

sample.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain check

I checked whether the combination of report date, client, and content item is unique for March 2026. If the total number of rows and unique combinations match, this supports the stated row grain.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,9841378


In [20]:
grain_check["difference"] = (
    grain_check["total_rows"] - grain_check["unique_grain_rows"]
)

grain_check

,total_rows,unique_grain_rows,difference
0,9841378,9841378,0


### Query 2 — Row count and date span

I checked the March 2026 slice to see how many rows it contains and what dates are covered. This confirms the size and time range of the data I am using.

In [21]:
date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

date_check

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — GSC availability

I checked how many March 2026 rows have GSC data available. I used `IS TRUE` so that only rows explicitly marked as available are counted.

In [22]:
gsc_availability = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


### Query 3 — GSC availability

I checked GSC availability for the March 2026 data. Out of 9,841,378 rows, 3,611,061 have GSC data available. I used `IS TRUE` so that only rows explicitly marked as available were counted.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation is that GSC data is not available for every row. In March 2026, only 3,611,061 out of 9,841,378 rows had GSC data available.

The data also does not tell me why a page is declining. It shows measured search and traffic signals, but it does not explain the reason behind a change.

I am also using March 2026 as the working month, so this slice may not represent longer-term or seasonal changes.

### Five features

**1. GSC impressions**  
Available at the decision moment because they are already recorded search impressions for the page.

**2. GSC clicks**  
Available at the decision moment because the clicks have already been recorded before making the decision.

**3. GSC average position**  
Available at the decision moment because it is calculated from the search data already collected.

**4. GA4 sessions**  
Available at the decision moment because the sessions are already recorded in the analytics data.

**5. GA4 engaged sessions**  
Available at the decision moment because the engagement information has already been recorded before the decision.

In [23]:
feature_frame = con.sql("""
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    LIMIT 10
""").df()

feature_frame

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


Some GA4 values are missing because GA4 data is not available for every row. I kept these values as missing instead of treating them as zero.

### Label / proxy

For this notebook I will use a simple decline proxy based on GSC clicks. A page is marked as declining when its clicks are zero in the observed period. This is only a proxy, not a causal explanation of why a page declined.

In [24]:
label_frame = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions,
        CASE
            WHEN gsc_data_available IS TRUE
                 AND gsc_clicks = 0
            THEN 1
            ELSE 0
        END AS declining_proxy
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    LIMIT 10000
""").df()

label_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,declining_proxy
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,1
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,1
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,0
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,1
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,1


In [25]:
label_frame["leaked_declining_feature"] = label_frame["declining_proxy"]

label_frame[[
    "gsc_clicks",
    "declining_proxy",
    "leaked_declining_feature"
]].head()

,gsc_clicks,declining_proxy,leaked_declining_feature
0,0,1,1
1,0,1,1
2,1,0,0
3,0,1,1
4,0,1,1


In [26]:
from sklearn.metrics import accuracy_score

honest_prediction = (label_frame["gsc_clicks"] == 0).astype(int)

honest_score = accuracy_score(
    label_frame["declining_proxy"],
    honest_prediction
)

leaked_score = accuracy_score(
    label_frame["declining_proxy"],
    label_frame["leaked_declining_feature"]
)

print("Honest score:", honest_score)
print("Leaked score:", leaked_score)

Honest score: 1.0
Leaked score: 1.0


In [27]:
label_frame = label_frame.drop(columns=["leaked_declining_feature"])

print("Leaked feature removed.")
print("Honest score:", honest_score)

Leaked feature removed.
Honest score: 1.0


### Leakage result

I deliberately added a feature derived directly from the label. The score became 1.0 because the feature was effectively the answer itself. This is data leakage, so the feature cannot be used for a real model.

I removed the leaked feature and kept the honest score instead. The leaked score should not be reported as a real model result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.